### Transform Customers Data

- Remove Records with NULL customer_id
- Remove Exact duplicate records
- Remove Duplicate Records based on create_timestamp
- CAST the columns to the correct Data Type
- Write Transformed Data to the Silver Schema

#### Remove Records with NULL customer_id

In [0]:
%sql
DESCRIBE EXTENDED gizmobox.bronze.py_customers;

In [0]:
# df = spark.sql('select * from gizmobox.bronze.py_customers');
# df = spark.table('gizmobox.bronze.py_customers');

df = spark.read.table('gizmobox.bronze.py_customers')

display(df)

In [0]:
# filtered_data = df.filter('customer_id is not NULL')

# display(filtered_data)

# filtered_data = df.filter(df.customer_id.isNotNull())

# display(filtered_data)


filtered_data = df.where(df.customer_id.isNotNull())

display(filtered_data);

In [0]:
%sql
select * from gizmobox.bronze.v_customers
where customer_id is not null;

#### Remove Exact duplicate records

In [0]:
%sql
select * from gizmobox.bronze.v_customers
where customer_id is not null 
order by customer_id;

In [0]:
df_distinct = filtered_data.distinct()
display(df_distinct)

In [0]:
%sql
select distinct * from gizmobox.bronze.v_customers
where customer_id is not null
order by customer_id;

In [0]:
%sql
create or replace temporary view tv_customers as
select distinct * from gizmobox.bronze.v_customers
where customer_id is not null
order by customer_id;
select * from v_customers

In [0]:
%sql
select * from tv_customers;

In [0]:
%sql
with distinct_id_and_timestamp as (
    select customer_id, max(created_timestamp) as time_stamp from tv_customers
    group by customer_id
)


select tv_customers.* from tv_customers 
inner join distinct_id_and_timestamp
on tv_customers.customer_id = distinct_id_and_timestamp.customer_id
and tv_customers.created_timestamp = distinct_id_and_timestamp.time_stamp;


#### Remove Duplicate Records based on create_timestamp

In [0]:
from pyspark.sql.functions import max

df_max_ts = df_distinct.groupBy('customer_id').agg(
    max('created_timestamp').alias('max_created_timestamp')
)

display(df_max_ts)

In [0]:
df_distinct_customer = df_distinct.join(
    df_max_ts,
    ((df_distinct.customer_id == df_max_ts.customer_id) & (df_distinct.created_timestamp == df_max_ts.max_created_timestamp)),
    'inner'
).select('*')


display(df_distinct_customer);

In [0]:
%sql
with distinct_id_and_timestamp as (
    select customer_id, max(created_timestamp) as time_stamp from tv_customers
    group by customer_id
)


select tv_customers.* from tv_customers 
inner join distinct_id_and_timestamp
on tv_customers.customer_id = distinct_id_and_timestamp.customer_id
and tv_customers.created_timestamp = distinct_id_and_timestamp.time_stamp;

#### CAST the columns to the correct Data Type

In [0]:
%sql
with distinct_id_and_timestamp as (
    select customer_id, max(created_timestamp) as time_stamp from tv_customers
    group by customer_id
)
select 
    CAST(t.created_timestamp AS TIMESTAMP) as created_timestamp,
    CAST(t.customer_id AS BIGINT) as customer_id,
    t.customer_name,
    CAST(t.date_of_birth AS DATE) as date_of_birth,
    t.email,
    CAST(t.member_since AS DATE) as member_since,
    t.telephone,
    t.file_path
from tv_customers t
inner join distinct_id_and_timestamp
on t.customer_id = distinct_id_and_timestamp.customer_id
and t.created_timestamp = distinct_id_and_timestamp.time_stamp;


#### Write Transformed Data to the Silver Schema or Write Data to the Delta Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gizmobox.silver.customers
as 
with distinct_id_and_timestamp as (
    select customer_id, max(created_timestamp) as time_stamp from tv_customers
    group by customer_id
)
select 
    CAST(t.created_timestamp AS TIMESTAMP) as created_timestamp,
    CAST(t.customer_id AS BIGINT) as customer_id,
    t.customer_name,
    CAST(t.date_of_birth AS DATE) as date_of_birth,
    t.email,
    CAST(t.member_since AS DATE) as member_since,
    t.telephone,
    t.file_path
from tv_customers t
inner join distinct_id_and_timestamp
on t.customer_id = distinct_id_and_timestamp.customer_id
and t.created_timestamp = distinct_id_and_timestamp.time_stamp;

In [0]:
%sql
select * from gizmobox.silver.customers

In [0]:
%sql
DESCRIBE TABLE EXTENDED gizmobox.silver.customers;

In [0]:
%sql 
describe schema extended gizmobox.bronze;

describe schema extended gizmobox.gold